In [ ]:
#!/usr/bin/env python3

import pandas as pd
import numpy as np
from pathlib import Path

INPUT_FILE = Path("grid_master_hourly_2030_2045.parquet")
OUTPUT_DIR = Path("system_impact")

def run_system_impact_analysis():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Reading master dataset...")
    df = pd.read_parquet(INPUT_FILE)

    df_main = df[df['weather_year_role'] == 'average_wind'].copy()

    wind_threshold = df['wind_mw'].quantile(0.1)
    print(f"Global low wind threshold (10th percentile): {wind_threshold:.2f} MW")

    df_main['residual_before_smr_mw'] = (
        df_main['demand_mw'] -
        df_main[['wind_mw', 'solar_mw', 'nuclear_existing_mw', 'biomass_mw', 'hydro_mw', 'other_mw']].sum(axis=1) -
        df_main['imports_net_baseline_mw']
    )

    df_main['residual_after_smr_mw'] = (
        df_main['residual_before_smr_mw'] - df_main['smr_total_delivered_mw']
    )

    df_main['gas_needed_before_mw'] = np.maximum(df_main['residual_before_smr_mw'], 0)
    df_main['gas_needed_after_mw'] = np.maximum(df_main['residual_after_smr_mw'], 0)

    df_main['gas_displacement_proxy_mw'] = (
        df_main['gas_needed_before_mw'] - df_main['gas_needed_after_mw']
    )

    df_main['surplus_after_smr_mw'] = np.where(
        df_main['residual_after_smr_mw'] < 0,
        -df_main['residual_after_smr_mw'],
        0
    )

    df_main['low_wind_flag'] = (
        df_main['wind_mw'] <= wind_threshold
    ).astype(int)

    hourly_cols = [
        "timestamp_utc", "year", "fes_scenario", "climate_member", "smr_case",
        "residual_before_smr_mw", "residual_after_smr_mw",
        "gas_needed_before_mw", "gas_needed_after_mw",
        "gas_displacement_proxy_mw", "surplus_after_smr_mw", "low_wind_flag"
    ]

    df_main[hourly_cols].to_parquet(
        OUTPUT_DIR / "system_impact_metrics_hourly_2030_2045.parquet",
        index=False
    )

    print("Hourly metrics saved.")

    annual_summary = df_main.groupby(
        ['year', 'fes_scenario', 'climate_member', 'weather_year_role', 'smr_case']
    ).agg(
        annual_gas_displacement_twh=(
            'gas_displacement_proxy_mw',
            lambda x: x.sum() / 1_000_000
        ),
        surplus_hours_count=(
            'surplus_after_smr_mw',
            lambda x: (x > 0).sum()
        ),
        low_wind_support_hours=(
            'low_wind_flag',
            'sum'
        )
    ).reset_index()

    annual_summary = annual_summary[[
        'year', 'fes_scenario', 'climate_member',
        'weather_year_role', 'smr_case',
        'annual_gas_displacement_twh',
        'surplus_hours_count',
        'low_wind_support_hours'
    ]]

    annual_summary.to_csv(
        OUTPUT_DIR / "system_impact_summary_annual_2030_2045.csv",
        index=False
    )

    print("Annual summary table saved.")

    period_summary = annual_summary.groupby(
        ['fes_scenario', 'smr_case']
    ).agg(
        cumulative_gas_displacement_twh=(
            'annual_gas_displacement_twh',
            'sum'
        ),
        average_residual_demand_reduction_mw=(
            'annual_gas_displacement_twh',
            'mean'
        )
    ).reset_index()

    period_summary.to_csv(
        OUTPUT_DIR / "system_impact_summary_period_2030_2045.csv",
        index=False
    )

    print("Period summary table saved.")

if __name__ == "__main__":
    run_system_impact_analysis()

In [ ]:
# check the output:

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("system_impact")

parquet_path = output_dir / "system_impact_metrics_hourly_2030_2045.parquet"
df_hourly = pd.read_parquet(parquet_path)

print("--- Hourly Metrics Preview (First 10 Rows) ---")
print(df_hourly.head(10))

print("\nColumn Names:\n", df_hourly.columns.tolist())

annual_path = output_dir / "system_impact_summary_annual_2030_2045.csv"
df_annual = pd.read_csv(annual_path)

print("\n--- Annual Summary Preview (First 5 Rows) ---")
print(df_annual.head(5))

period_path = output_dir / "system_impact_summary_period_2030_2045.csv"
df_period = pd.read_csv(period_path)

print("\n--- Period Summary Content ---")
print(df_period)

In [ ]:
# hourly datasets check (2035, 2036, 2037):

In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path("system_impact")
parquet_path = output_dir / "system_impact_metrics_hourly_2030_2045.parquet"

try:
    df = pd.read_parquet(parquet_path)

    print("Available columns:", df.columns.tolist())
    print("=" * 60)

    for year in [2035, 2036, 2037]:
        print(f"\n--- First 10 Rows for Year {year} ---")

        df_year = df[df['year'] == year]
        print(df_year.head(10))

except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
# gas_reference check:

In [ ]:
import pandas as pd
from pathlib import Path

parquet_path = "grid_master_hourly_2030_2045.parquet"
df = pd.read_parquet(parquet_path)

if "gas_reference_mw" in df.columns:
    print("gas_reference_mw Description:")
    print(df["gas_reference_mw"].describe())

    all_zero = (df["gas_reference_mw"] == 0).all()
    print(f"\nIs this column entirely zero: {all_zero}")

    non_zero_count = (df["gas_reference_mw"] != 0).sum()
    print(f"Number of non-zero elements: {non_zero_count}")

else:
    print("The column 'gas_reference_mw' was not found in the dataset.")

In [ ]:
# the others checking:

In [ ]:
import pandas as pd
from pathlib import Path

parquet_path = Path("system_impact/system_impact_metrics_hourly_2030_2045.parquet")
csv_path = Path("system_impact/system_impact_summary_annual_2030_2045.csv")

print("=== Checking Hourly Data ===")

try:
    df_hourly = pd.read_parquet(parquet_path)

    cols = [
        'gas_needed_before_mw',
        'gas_needed_after_mw',
        'gas_displacement_proxy_mw'
    ]

    print(df_hourly[cols].describe())

    for col in cols:
        non_zero = (df_hourly[col] != 0).sum()
        print(f"{col} - Number of non-zero elements: {non_zero}")

    print("\n=== Sample Non-Zero Data ===")

    nonzero_df = df_hourly[
        (df_hourly['gas_needed_before_mw'] != 0) |
        (df_hourly['gas_needed_after_mw'] != 0) |
        (df_hourly['gas_displacement_proxy_mw'] != 0)
    ]

    print(nonzero_df.head(5))

except Exception as e:
    print(f"Error reading Parquet file: {e}")

print("\n=== Checking Annual Data ===")

try:
    df_annual = pd.read_csv(csv_path)

    target_col = 'annual_gas_displacement_twh'

    if target_col in df_annual.columns:
        print(df_annual[['year', target_col]].describe())

        print("\n=== Annual Detailed Data ===")
        print(df_annual[['year', target_col]].head(10))

    else:
        print(f"The column {target_col} was not found in the Annual CSV file.")

except Exception as e:
    print(f"Error reading CSV file: {e}")

In [ ]:
import pandas as pd

file_path = "system_impact/system_impact_summary_annual_2030_2045.csv"

try:
    df_annual = pd.read_csv(file_path)

    print("=" * 40)
    print(" 1. Frequency Statistics for Categorical Fields")
    print("=" * 40)

    print("\n--- weather_year_role Statistics ---")
    print(df_annual['weather_year_role'].value_counts(dropna=False))

    print("\n--- smr_case Statistics ---")
    print(df_annual['smr_case'].value_counts(dropna=False))

    print("\n" + "=" * 40)
    print(" 2. Summary Statistics for Numeric Fields")
    print("=" * 40)

    numeric_cols = [
        'annual_gas_displacement_twh',
        'surplus_hours_count',
        'low_wind_support_hours'
    ]

    print(df_annual[numeric_cols].describe())

    print("\n" + "=" * 40)
    print(" 3. Missing Value Check")
    print("=" * 40)

    print(df_annual.isnull().sum())

except FileNotFoundError:
    print(f"File not found: {file_path}. Please ensure the path is correct.")